## AgentCore Runtime의 End-to-End Stateless MCP Server

이 예제에서는 AgentCore Runtime에 배포된 전체 MCP server capability(MCP Spec 기반)를 보여줍니다.

예를 들어 MCP native resource, prompt, tool을 활용할 수 있는 MCP server를 AgentCore Runtime에 생성할 때 유용합니다.

이 튜토리얼에서는 다음 내용을 학습합니다.

* tool, prompt, resource가 포함된 MCP server를 생성하는 방법
* AgentCore Runtime에 배포하는 방법
* 배포된 server를 호출하는 방법

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Runtime에 Tool, Prompt, Resource 호스팅                   |
| Tool 유형           | MCP server                                                |
| 튜토리얼 구성 요소  | AgentCore Runtime에 호스팅, MCP server 생성               |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 중급                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 MCP Client          |

### 튜토리얼 아키텍처

이 튜토리얼에서는 이 예제를 AgentCore Runtime에 배포하는 방법을 설명합니다.

<img src="img/architecture.png" style="width: 80%;">

이 튜토리얼 Notebook에서는 하나의 agent를 구축합니다. 먼저 네 개의 tool과 함께 agent를 AgentCore Runtime에 배포합니다. 그런 다음 prompt를 추가하도록 업데이트하고 마지막으로 resource를 배포하도록 다시 업데이트합니다.

이제 시작해 보겠습니다.

시작하려면 필수 dependency를 설치한 다음 kernel을 다시 시작합니다.

In [ ]:
!pip install -qU -r requirements.txt

script가 helpers 폴더에 액세스할 수 있도록 다음 path를 추가합니다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

### MCP Server 생성

먼저 tool만 포함된 MCP Server를 생성하고 이후 다른 기능을 추가합니다.

이 MCP는 네 개의 MCP tool로 구성됩니다.

- **add_expense**: 사용자 지출 추가
- **add_income**: 사용자 수입 추가
- **set_budget**: 지출 한도를 설정
- **get_balance**: 월간 잔액 가져오기

데이터는 DynamoDB table에 영구 저장되므로 이 튜토리얼의 다음 단계에서도 사용할 수 있습니다. 이렇게 하면 MCP server는 stateless 상태를 유지하고 사용자 정보는 DynamoDB table에 저장됩니다.

다음 코드를 실행하여 MCP server 로컬 파일과 requirements 파일을 생성합니다.

In [ ]:
%%writefile agents/mcp_e2e_stateless_server.py
import os
from mcp.server.fastmcp import FastMCP
from dynamo_utils import FinanceDB

mcp = FastMCP(name="Stateless-MCP-Server",
              host="0.0.0.0", 
              stateless_http=True) # Stateless mode - session persistence 없음

_region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
db = FinanceDB(region_name=_region)

@mcp.tool()
def add_expense(user_alias: str, amount: float, description: str, category: str = "other") -> str:
    """Add a new expense transaction
    
    Args:
        user_alias: User identifier
        amount: Expense amount (positive number)
        description: Description of the expense
        category: Expense category (food, transport, entertainment, bills, other)
    """
    return db.add_transaction(user_alias, "expense", -abs(amount), description, category)

@mcp.tool()
def add_income(user_alias: str, amount: float, description: str, source: str = "salary") -> str:
    """Add a new income transaction
    
    Args:
        user_alias: User identifier
        amount: Income amount (positive number)
        description: Description of the income
        source: Income source (salary, freelance, investment, other)
    """
    return db.add_transaction(user_alias, "income", abs(amount), description, source)

@mcp.tool()
def set_budget(user_alias: str, category: str, monthly_limit: float) -> str:
    """Set monthly budget limit for a category
    
    Args:
        user_alias: User identifier
        category: Budget category (food, transport, entertainment, bills, other)
        monthly_limit: Monthly spending limit for this category
    """
    return db.set_budget(user_alias, category, monthly_limit)

@mcp.tool()
def get_balance(user_alias: str) -> str:
    """Get current account balance
    
    Args:
        user_alias: User identifier
    """
    balance_data = db.get_balance(user_alias)
    return f"Balance: ${balance_data['balance']:.2f}\nTotal Income: ${balance_data['income']:.2f}\nTotal Expenses: ${balance_data['expenses']:.2f}"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")


In [ ]:
%%writefile agents/requirements.txt
#fastmcp>=2.10.0
#mcp==1.19.0
mcp
bedrock-agentcore

`dynamo_utils` 파일을 agents 폴더로 복사합니다.

In [ ]:
!cp ../helpers/dynamo_utils.py agents/dynamo_utils.py

MCP에서 사용할 DynamoDB table을 생성합니다.

In [ ]:
import boto3

from helpers.dynamo_utils import FinanceDB

region = boto3.session.Session().region_name
db = FinanceDB(region_name=region)
result = db.create_table()
print(f"Region: {region}")
print(result)

#### 인증을 위한 Cognito User Pool 생성

MCP Server에서 인증을 보장하도록 Cognito user pool을 생성합니다.

In [ ]:
from helpers.utils import get_or_create_cognito_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = get_or_create_cognito_pool()  # 이 output cell에서 bearer token을 가져옴
print("Cognito setup completed ✓")

AgentCore execution role을 생성합니다.

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

execution_role_arn_mcp = create_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)

#### AgentCore Runtime에서 MCP Server 구성 및 시작

이 예제를 AgentCore Runtime에 배포합니다.

In [ ]:
# library 가져오기
import json
import boto3
from boto3.session import Session

# boto session 가져오기
boto_session = Session()

sts = boto3.client("sts")
response = sts.get_caller_identity()
account_id = response["Account"]
region = boto_session.region_name

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_mcp_agent = Runtime()
aws_agent_name = "mcp_e2e_stateless_server"

# 배포 구성
response_aws_mcp_agent = agentcore_runtime_mcp_agent.configure(
    entrypoint="agents/mcp_e2e_stateless_server.py",
    execution_role=execution_role_arn_mcp,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="MCP",
    deployment_type="direct_code_deploy",
    runtime_type="PYTHON_3_13",
)

print("Configuration completed:", response_aws_mcp_agent)

In [ ]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("Launch completed:", launch_result_mcp.agent_arn)

mcp_arn = launch_result_mcp.agent_arn

In [ ]:
status_response = agentcore_runtime_mcp_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

#### MCP Server 테스트

tool이 포함된 배포 MCP의 첫 번째 version을 테스트합니다.

먼저 새 JWT token을 가져옵니다.

In [ ]:
ac_runtime_name = launch_result_mcp.agent_id

mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{ac_runtime_name}/invocations?qualifier=DEFAULT&accountId={account_id}"

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
}

MCP를 호출합니다.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

print(f"Invoking: {mcp_url} \n")
async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
    read_stream,
    write_stream,
    _,
):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        tool_result = await session.list_tools()
        print("\n📋 Available MCP Tools:")
        print("=" * 50)
        for tool in tool_result.tools:
            print(f"🔧 {tool.name}: {tool.description}")

        print("\n✅ MCP tool testing completed!")

DynamoDB Table에 데이터를 추가하는 몇 가지 작업을 수행합니다.

In [ ]:
async def invoke_tool(tool_name: str, tool_params: dict):
    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            try:
                print(f"\n➕ Testing tool: {tool_name}()...")
                tool_result = await session.call_tool(name=tool_name, arguments=tool_params)
                print(f"   Result: {tool_result.content[0].text}")
            except Exception as e:
                print(f"   Error: {e}")

In [ ]:
tool_name = "add_expense"
tool_arguments = {
    "user_alias": "me",
    "amount": 30.5,
    "description": "Today's dinner",
    "category": "food",
}

await invoke_tool(tool_name, tool_arguments)

tool_arguments = {
    "user_alias": "me",
    "amount": 15,
    "description": "Today's breakfast",
    "category": "food",
}

await invoke_tool(tool_name, tool_arguments)

tool_arguments = {
    "user_alias": "me",
    "amount": 120,
    "description": "Energy",
    "category": "bills",
}

await invoke_tool(tool_name, tool_arguments)

In [ ]:
tool_name = "add_income"
tool_arguments = {
    "user_alias": "me",
    "amount": 1000,
    "description": "paycheck",
    "source": "salary",
}

await invoke_tool(tool_name, tool_arguments)

In [ ]:
tool_name = "set_budget"
tool_arguments = {"user_alias": "me", "category": "bills", "monthly_limit": 100}

await invoke_tool(tool_name, tool_arguments)

In [ ]:
tool_name = "get_balance"
tool_arguments = {"user_alias": "me"}

await invoke_tool(tool_name, tool_arguments)

table에 데이터가 추가되었으므로 예제를 계속 진행하여 MCP에 Prompt를 추가합니다.

---

#### Prompt를 추가하도록 Runtime 변경

기능을 추가하도록 MCP 코드를 일부 변경합니다.

In [ ]:
%%writefile agents/mcp_e2e_stateless_server.py
import os
from mcp.server.fastmcp import FastMCP
from mcp.types import PromptMessage, TextContent
from dynamo_utils import FinanceDB

mcp = FastMCP(name="Stateless-MCP-Server",
              host="0.0.0.0", 
              stateless_http=True) # Stateless mode - session persistence 없음

_region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
db = FinanceDB(region_name=_region)

@mcp.tool()
def add_expense(user_alias: str, amount: float, description: str, category: str = "other") -> str:
    """Add a new expense transaction
    
    Args:
        user_alias: User identifier
        amount: Expense amount (positive number)
        description: Description of the expense
        category: Expense category (food, transport, entertainment, bills, other)
    """
    return db.add_transaction(user_alias, "expense", -abs(amount), description, category)

@mcp.tool()
def add_income(user_alias: str, amount: float, description: str, source: str = "salary") -> str:
    """Add a new income transaction
    
    Args:
        user_alias: User identifier
        amount: Income amount (positive number)
        description: Description of the income
        source: Income source (salary, freelance, investment, other)
    """
    return db.add_transaction(user_alias, "income", abs(amount), description, source)

@mcp.tool()
def set_budget(user_alias: str, category: str, monthly_limit: float) -> str:
    """Set monthly budget limit for a category
    
    Args:
        user_alias: User identifier
        category: Budget category (food, transport, entertainment, bills, other)
        monthly_limit: Monthly spending limit for this category
    """
    return db.set_budget(user_alias, category, monthly_limit)

@mcp.tool()
def get_balance(user_alias: str) -> str:
    """Get current account balance
    
    Args:
        user_alias: User identifier
    """
    balance_data = db.get_balance(user_alias)
    return f"Balance: ${balance_data['balance']:.2f}\nTotal Income: ${balance_data['income']:.2f}\nTotal Expenses: ${balance_data['expenses']:.2f}"

@mcp.prompt()
def budget_analysis(user_alias: str, time_period: str = "current_month") -> PromptMessage:
    """Analyze spending patterns and budget performance

    Args:
        user_alias: User identifier
        time_period: Time period to analyze (current_month, last_month, last_3_months)
    """
    # DynamoDB에서 현재 지출 데이터 가져오기
    transactions = db.get_transactions(user_alias)
    budgets = db.get_budgets(user_alias)
    
    current_spending = {}
    for transaction in transactions:
        if transaction["type"] == "expense":
            category = transaction["category"]
            current_spending[category] = current_spending.get(category, 0) + abs(float(transaction["amount"]))

    spending_summary = "\n".join([f"- {cat}: ${amount:.2f}" for cat, amount in current_spending.items()])
    budget_summary = "\n".join([f"- {budget['category']}: ${float(budget['monthly_limit']):.2f}/month" for budget in budgets])

    return PromptMessage(
        role="user",
        content=TextContent(
            type="text",
            text=f"""Please analyze my financial data for {time_period} and provide insights:

CURRENT SPENDING BY CATEGORY:
{spending_summary or "No expenses recorded"}

BUDGET LIMITS:
{budget_summary or "No budgets set"}

Please provide:
1. Budget vs actual spending comparison
2. Categories where I'm overspending
3. Recommendations for better budget management
4. Trends and patterns you notice
"""
        )
    )

@mcp.prompt()
def savings_plan(user_alias: str, target_amount: float, target_months: int = 12) -> PromptMessage:
    """Generate a personalized savings plan

    Args:
        user_alias: User identifier
        target_amount: Target savings amount
        target_months: Number of months to reach the target (default 12)
    """
    # DynamoDB에서 현재 재무 상태 계산
    balance_data = db.get_balance(user_alias)
    total_income = balance_data['income']
    total_expenses = balance_data['expenses']
    current_balance = balance_data['balance']

    monthly_target = target_amount / target_months

    return PromptMessage(
        role="user",
        content=TextContent(
            type="text",
            text=f"""Help me create a savings plan based on my financial situation:

SAVINGS GOAL:
- Target Amount: ${target_amount:.2f}
- Time Frame: {target_months} months
- Monthly Savings Needed: ${monthly_target:.2f}

CURRENT FINANCIAL SITUATION:
- Current Balance: ${current_balance:.2f}
- Total Income: ${total_income:.2f}
- Total Expenses: ${total_expenses:.2f}

Please provide:
1. Assessment of whether this savings goal is realistic
2. Specific strategies to reduce expenses
3. Ways to increase income if needed
4. Monthly action plan to reach the target
5. Emergency fund recommendations
"""
        )
    )

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

prompt를 포함하여 다시 배포합니다.

In [ ]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("Launch completed:", launch_result_mcp.agent_arn)

mcp_arn = launch_result_mcp.agent_arn

#### 새 기능 테스트

MCP server에 방금 추가한 새 기능을 테스트합니다.

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
}

In [ ]:
from IPython.display import display, Markdown

print(f"Invoking: {mcp_url} \n")
async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
    read_stream,
    write_stream,
    _,
):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        # prompt 목록 표시
        prompts = await session.list_prompts()
        for p in prompts:
            display(Markdown(f"prompts: {p}"))

        # "budget_analysis" prompt 호출
        result = await session.get_prompt("budget_analysis", {"user_alias": "me"})
        display(Markdown(f"budget_analysis: {result}"))

        # "savings_plan" prompt 호출
        result = await session.get_prompt("savings_plan", {"user_alias": "me", "target_amount": "800"})
        display(Markdown(f"savings_plan: {result}"))

---

#### Resource를 추가하도록 Runtime 변경

MCP 코드를 일부 변경하여 기능을 더 추가하고 완성합니다.

In [ ]:
%%writefile agents/mcp_e2e_stateless_server.py
import os
import json
from datetime import datetime
from mcp.server.fastmcp import FastMCP
from mcp.types import PromptMessage, TextContent
from dynamo_utils import FinanceDB


mcp = FastMCP(name="Stateless-MCP-Server",
              host="0.0.0.0", 
              stateless_http=True) # Stateless mode - session persistence 없음

_region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
db = FinanceDB(region_name=_region)

@mcp.tool()
def add_expense(user_alias: str, amount: float, description: str, category: str = "other") -> str:
    """Add a new expense transaction
    
    Args:
        user_alias: User identifier
        amount: Expense amount (positive number)
        description: Description of the expense
        category: Expense category (food, transport, entertainment, bills, other)
    """
    return db.add_transaction(user_alias, "expense", -abs(amount), description, category)

@mcp.tool()
def add_income(user_alias: str, amount: float, description: str, source: str = "salary") -> str:
    """Add a new income transaction
    
    Args:
        user_alias: User identifier
        amount: Income amount (positive number)
        description: Description of the income
        source: Income source (salary, freelance, investment, other)
    """
    return db.add_transaction(user_alias, "income", abs(amount), description, source)

@mcp.tool()
def set_budget(user_alias: str, category: str, monthly_limit: float) -> str:
    """Set monthly budget limit for a category
    
    Args:
        user_alias: User identifier
        category: Budget category (food, transport, entertainment, bills, other)
        monthly_limit: Monthly spending limit for this category
    """
    return db.set_budget(user_alias, category, monthly_limit)

@mcp.tool()
def get_balance(user_alias: str) -> str:
    """Get current account balance
    
    Args:
        user_alias: User identifier
    """
    balance_data = db.get_balance(user_alias)
    return f"Balance: ${balance_data['balance']:.2f}\nTotal Income: ${balance_data['income']:.2f}\nTotal Expenses: ${balance_data['expenses']:.2f}"

@mcp.prompt()
def budget_analysis(user_alias: str, time_period: str = "current_month") -> PromptMessage:
    """Analyze spending patterns and budget performance

    Args:
        user_alias: User identifier
        time_period: Time period to analyze (current_month, last_month, last_3_months)
    """
    # DynamoDB에서 현재 지출 데이터 가져오기
    transactions = db.get_transactions(user_alias)
    budgets = db.get_budgets(user_alias)
    
    current_spending = {}
    for transaction in transactions:
        if transaction["type"] == "expense":
            category = transaction["category"]
            current_spending[category] = current_spending.get(category, 0) + abs(float(transaction["amount"]))

    spending_summary = "\n".join([f"- {cat}: ${amount:.2f}" for cat, amount in current_spending.items()])
    budget_summary = "\n".join([f"- {budget['category']}: ${float(budget['monthly_limit']):.2f}/month" for budget in budgets])

    return PromptMessage(
        role="user",
        content=TextContent(
            type="text",
            text=f"""Please analyze my financial data for {time_period} and provide insights:

CURRENT SPENDING BY CATEGORY:
{spending_summary or "No expenses recorded"}

BUDGET LIMITS:
{budget_summary or "No budgets set"}

Please provide:
1. Budget vs actual spending comparison
2. Categories where I'm overspending
3. Recommendations for better budget management
4. Trends and patterns you notice
"""
        )
    )

@mcp.prompt()
def savings_plan(user_alias: str, target_amount: float, target_months: int = 12) -> PromptMessage:
    """Generate a personalized savings plan

    Args:
        user_alias: User identifier
        target_amount: Target savings amount
        target_months: Number of months to reach the target (default 12)
    """
    # DynamoDB에서 현재 재무 상태 계산
    balance_data = db.get_balance(user_alias)
    total_income = balance_data['income']
    total_expenses = balance_data['expenses']
    current_balance = balance_data['balance']

    monthly_target = target_amount / target_months

    return PromptMessage(
        role="user",
        content=TextContent(
            type="text",
            text=f"""Help me create a savings plan based on my financial situation:

SAVINGS GOAL:
- Target Amount: ${target_amount:.2f}
- Time Frame: {target_months} months
- Monthly Savings Needed: ${monthly_target:.2f}

CURRENT FINANCIAL SITUATION:
- Current Balance: ${current_balance:.2f}
- Total Income: ${total_income:.2f}
- Total Expenses: ${total_expenses:.2f}

Please provide:
1. Assessment of whether this savings goal is realistic
2. Specific strategies to reduce expenses
3. Ways to increase income if needed
4. Monthly action plan to reach the target
5. Emergency fund recommendations
"""
        )
    )

@mcp.resource("finance://monthly/{user_alias}")
def get_monthly_summary(user_alias: str) -> str:
    """Get monthly financial summary as JSON"""
    now = datetime.now()
    current_month_start = now.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    
    # DynamoDB에서 transaction 가져오기
    all_transactions = db.get_transactions(user_alias)
    monthly_transactions = [
        t for t in all_transactions 
        if datetime.fromisoformat(t["date"]) >= current_month_start
    ]
    
    monthly_income = sum(float(t["amount"]) for t in monthly_transactions if t["type"] == "income")
    monthly_expenses = sum(abs(float(t["amount"])) for t in monthly_transactions if t["type"] == "expense")
    
    # category별 지출 grouping
    expenses_by_category = {}
    for t in monthly_transactions:
        if t["type"] == "expense":
            category = t["category"]
            expenses_by_category[category] = expenses_by_category.get(category, 0) + abs(float(t["amount"]))
    
    summary = {
        "user": user_alias,
        "month": now.strftime("%Y-%m"),
        "income": monthly_income,
        "expenses": monthly_expenses,
        "net": monthly_income - monthly_expenses,
        "expenses_by_category": expenses_by_category,
        "transaction_count": len(monthly_transactions),
        "generated_at": datetime.now().isoformat()
    }
    
    return json.dumps(summary, indent=2)


@mcp.resource("finance://budgets/{user_alias}")
def get_budget_status(user_alias: str) -> str:
    """Get current budget status and performance as JSON"""
    # DynamoDB에서 데이터 가져오기
    all_transactions = db.get_transactions(user_alias)
    all_budgets = db.get_budgets(user_alias)
    
    budget_status = {}
    
    # category별 이번 달 지출 계산
    now = datetime.now()
    current_month_start = now.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    
    monthly_spending = {}
    for transaction in all_transactions:
        if (transaction["type"] == "expense" and 
            datetime.fromisoformat(transaction["date"]) >= current_month_start):
            category = transaction["category"]
            monthly_spending[category] = monthly_spending.get(category, 0) + abs(float(transaction["amount"]))
    
    # budget과 비교
    for budget in all_budgets:
        category = budget["category"]
        budget_limit = float(budget["monthly_limit"])
        spent = monthly_spending.get(category, 0)
        remaining = budget_limit - spent
        usage_percent = (spent / budget_limit) * 100 if budget_limit > 0 else 0
        
        budget_status[category] = {
            "budget_limit": budget_limit,
            "spent_this_month": spent,
            "remaining": remaining,
            "usage_percent": usage_percent,
            "status": "over_budget" if spent > budget_limit else "within_budget",
            "set_date": budget["set_date"]
        }
    
    # 지출은 있지만 budget이 없는 category 추가
    for category, spent in monthly_spending.items():
        if category not in budget_status:
            budget_status[category] = {
                "budget_limit": None,
                "spent_this_month": spent,
                "remaining": None,
                "usage_percent": None,
                "status": "no_budget_set",
                "set_date": None
            }
    
    return json.dumps({
        "user": user_alias,
        "month": now.strftime("%Y-%m"),
        "budget_status": budget_status,
        "generated_at": datetime.now().isoformat()
    }, indent=2)

@mcp.resource("finance://import/{user_alias}/{filename}")
def import_expenses_from_file(user_alias: str, filename: str) -> str:
    """Import expenses from a local text file and insert into DynamoDB
    
    File format: expense,category,value (one per line)
    Example: Lunch,food,25.50
    
    Args:
        user_alias: User identifier
        filename: Name of the file to import (must be in agents folder)
    """
    import os
    import json
    
    try:
        file_path = filename  # filename을 직접 사용
        
        if not os.path.exists(file_path):
            return json.dumps({
                "error": f"File {filename} not found in agents directory",
                "current_directory": os.getcwd(),
                "imported_count": 0
            }, indent=2)
        
        imported_expenses = []
        error_lines = []
        
        with open(file_path, 'r') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                try:
                    parts = [part.strip() for part in line.split(',')]
                    if len(parts) != 3:
                        error_lines.append(f"Line {line_num}: Invalid format - {line}")
                        continue
                    
                    description, category, amount_str = parts
                    amount = float(amount_str)
                    
                    result = db.add_transaction(user_alias, "expense", -abs(amount), description, category)
                    imported_expenses.append({
                        "description": description,
                        "category": category,
                        "amount": amount,
                        "result": result
                    })
                    
                except ValueError:
                    error_lines.append(f"Line {line_num}: Invalid amount - {line}")
                except Exception as e:
                    error_lines.append(f"Line {line_num}: Error - {str(e)}")
        
        return json.dumps({
            "user": user_alias,
            "filename": filename,
            "imported_count": len(imported_expenses),
            "error_count": len(error_lines),
            "imported_expenses": imported_expenses,
            "errors": error_lines,
            "generated_at": datetime.now().isoformat()
        }, indent=2)
        
    except Exception as e:
        return json.dumps({
            "error": f"Failed to process file: {str(e)}",
            "imported_count": 0
        }, indent=2)

if __name__ == "__main__":
    mcp.run(transport="streamable-http")


다시 배포합니다.

In [ ]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("Launch completed:", launch_result_mcp.agent_arn)

mcp_arn = launch_result_mcp.agent_arn

#### Resource 테스트

MCP server가 새 resource를 반환할 수 있는지 테스트합니다.

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
}

In [ ]:
from IPython.display import display, Markdown

print(f"Invoking: {mcp_url} \n")
async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
    read_stream,
    write_stream,
    _,
):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        print("\n--- Monthly Summary ---")
        user_alias = "me"
        result = await session.read_resource(f"finance://monthly/{user_alias}")
        print(result.contents[0].text)

        print("\n--- Budget ---")
        user_alias = "me"
        result = await session.read_resource(f"finance://budgets/{user_alias}")
        print(result.contents[0].text)

파일로 테스트합니다.

In [ ]:
print(f"Invoking resource with a local File: {mcp_url} \n")
async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
    read_stream,
    write_stream,
    _,
):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        print("\n--- New Expenses ---")
        user_alias = "me"
        result = await session.read_resource(f"finance://import/{user_alias}/expenses.txt")
        print(result.contents[0].text)

축하합니다. 이 lab을 완료했습니다.

---

### 리소스 정리(선택 사항)

In [ ]:
from pathlib import Path
from bedrock_agentcore_starter_toolkit.operations.runtime.destroy import (
    destroy_bedrock_agentcore,
)

print("🚀 Starting Runtime cleanup...")
destroy_bedrock_agentcore(config_path=Path(".bedrock_agentcore.yaml"), agent_name=aws_agent_name)

In [ ]:
from helpers.utils import delete_agentcore_runtime_execution_role

# execution role 삭제
print("  🗑️  Deleting Agent execution role...")
delete_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)
print("  ✅ Execution role deleted")

In [ ]:
from helpers.utils import (
    cleanup_cognito_resources,
    delete_cognito_secret,
    get_cognito_secret,
)

# Cognito와 secret 정리
print("  🗑️  Cleaning up Cognito resources...")
cs = json.loads(get_cognito_secret())
cleanup_cognito_resources(cognito_config.get("pool_id"))
print("  ✅ Cognito resources cleaned up")

print("  🗑️  Deleting customer support secret...")
delete_cognito_secret()
print("  ✅ Customer support secret deleted")

In [ ]:
# DynamoDB Table 삭제
db.delete_table()

In [ ]:
from helpers.utils import local_file_cleanup

print("📁 Starting Local Files cleanup...")
local_file_cleanup()